In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
#importing libraies

In [ ]:
df = pd.read_csv("animeEda.csv")
#Reading file
df

In [ ]:
df.head()
#load data


In [ ]:
df.tail()


In [ ]:
df.info()
df.describe()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
#changing ep datatype to int
df["episodes"] = pd.to_numeric(df["episodes"], errors="coerce")

df["episodes"] = df["episodes"].fillna(df["episodes"].mean())

df["episodes"] = df["episodes"].astype(int)
df

In [ ]:
#sorting rating
df = df.sort_values(by="rating", ascending=False)
df

In [ ]:
#creating rank
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# remove rows where rating is missing

# sort by rating
df = df.sort_values(by="rating", ascending=False)
df

In [ ]:


#sorting
df = df.sort_values(by="rating", ascending=False)
#rank
df["rank"] = df["rating"].rank(method="dense", ascending=False)
df

In [ ]:
df=df.dropna()
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df

In [ ]:
# split genres for top rated animes
genre_df = df.assign(genre=df["genre"].str.split(","))
genre_df = genre_df.explode("genre")
genre_df["genre"] = genre_df["genre"].str.strip()
top_genre = genre_df.sort_values("rating", ascending=False).groupby("genre").head(1)
top_genre[["genre","name","rating"]]

In [ ]:
# split genre for Most Popular Anime
genre_df = df.assign(genre=df["genre"].str.split(","))


genre_df = genre_df.explode("genre")


genre_df["genre"] = genre_df["genre"].str.strip()

popular_genre = genre_df.sort_values("members", ascending=False).groupby("genre").head(1)

popular_genre[["genre","name","members"]]

In [ ]:
# Most Popular Anime in Each Genre
genre_df = df.assign(genre=df["genre"].str.split(","))

genre_df = genre_df.explode("genre")

genre_df["genre"] = genre_df["genre"].str.strip()

popular_unique = genre_df.loc[genre_df.groupby("genre")["members"].idxmax()]

# combine genres for same anime
combined = popular_unique.groupby("name").agg({
    "genre": lambda x: ", ".join(sorted(set(x))),
    "members": "first"
}).reset_index()

# sort by popularity
combined = combined.sort_values("members", ascending=False)

combined

In [ ]:
#Best anime for kinds
kids_anime = df[df["genre"].str.contains("Kids", case=False, na=False)]
top_kids = kids_anime.sort_values(by="rating", ascending=False).head(10)
top_kids[["name","genre","rating"]]

In [ ]:
#Best anime for teenager
teen_anime = df[df["genre"].str.contains("Shounen|School|Action", case=False, na=False)]
top_teen = teen_anime.sort_values(by="rating", ascending=False).head(10)
top_teen[["name","genre","rating"]]

In [ ]:
#Best Anime for Adults
adult_anime = df[df["genre"].str.contains("Psychological|Seinen|Thriller", na=False)]
adult_anime.sort_values("rating", ascending=False)[["name","genre","rating"]].head(10)

In [ ]:
#Compare between kids/teenaggers/adults
def age_group(genre):
    if "Kids" in str(genre):
        return "Kids"
    elif "Shounen" in str(genre) or "School" in str(genre) or "Action" in str(genre):
        return "Teen"
    elif "Seinen" in str(genre) or "Psychological" in str(genre) or "Thriller" in str(genre):
        return "Adult"
    else:
        return "General"

df["age_group"] = df["genre"].apply(age_group)
top_age = df.sort_values("rating", ascending=False).groupby("age_group").head(5)
top_age[["age_group","name","rating"]]

In [ ]:
df = df.reset_index(drop=True)
df

In [ ]:
df.to_csv("anime_cleaned.csv", index=False)

In [ ]:
#Creating EDA
sns.countplot(x='type', data=df)
plt.title("Anime Type Distribution")
plt.show()

In [ ]:
sns.histplot(df["rating"], bins=20)
plt.title("Rating Distribution")
plt.show()

In [ ]:
top_rated = df.sort_values("rating", ascending=False).head(10)
sns.barplot(x="rating", y="name", data=top_rated)
plt.title("Top Rated Anime")
plt.show()

In [ ]:
top10_genre = top_genre.sort_values("rating", ascending=False).head(10)

plt.figure(figsize=(8,6))
sns.barplot(x="rating", y="genre", data=top10_genre)

plt.title("Top Rated Anime by Genre")
plt.show()

In [ ]:
popular = df.sort_values("members", ascending=False).head(5)
sns.barplot(x="members", y="name", data=popular)
plt.title("Most Popular Anime")
plt.show()

In [ ]:
top10_popular = popular_genre.sort_values("members", ascending=False).head(10)

plt.figure(figsize=(8,6))
sns.barplot(x="members", y="genre", data=top10_popular)

plt.title("Most Popular Anime by Genre")
plt.show()

In [ ]:
df.groupby("type")["rating"].mean().plot(kind="bar")
plt.title("Average Rating by Type")
plt.show()

In [ ]:
genre = df['genre'].str.split(',', expand=True).stack()

genre.value_counts().head(10).plot(kind="bar")
plt.title("Top Genres")
plt.show()

In [ ]:
# create label with name and genre
combined["label"] = combined["name"] + " (" + combined["genre"] + ")"

# sort by popularity
top10 = combined.sort_values("members", ascending=False).head(10)

plt.figure(figsize=(10,6))

sns.barplot(x="members", y="label", data=top10)

plt.title("Top 10 Most Popular Anime with Genres")
plt.xlabel("Members")
plt.ylabel("Anime + Genre")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.barplot(x="rating", y="name", data=top_kids)

plt.title("Top 10 Anime for Kids")
plt.xlabel("Rating")
plt.ylabel("Anime Name")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.barplot(x="rating", y="name", data=top_teen)

plt.title("Top 10 Anime for Teen Audience")
plt.xlabel("Rating")
plt.ylabel("Anime Name")

plt.show()

In [ ]:
top_adult = adult_anime.sort_values("rating", ascending=False).head(10)

sns.barplot(x="rating", y="name", data=top_adult)

plt.title("Top Anime for Adult Audience")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.barplot(x="rating", y="name", hue="age_group", data=top_age)

plt.title("Top Anime by Age Group")

plt.show()

In [ ]:
long_anime = df.sort_values("episodes", ascending=False).head(10)

sns.barplot(x="episodes", y="name", data=long_anime)
plt.title("Longest Anime")
plt.show()

In [ ]:
df["episodes"] = pd.to_numeric(df["episodes"], errors="coerce")
df["episodes"] = df["episodes"].fillna(df["episodes"].mean())
df.groupby("type")["episodes"].mean().plot(kind="bar")
plt.title("Average Episodes by Type")
plt.show()

In [ ]:
sns.histplot(df["members"], bins=30)
plt.title("Members Distribution")
plt.show()

In [ ]:
df.groupby("type")["members"].sum().plot(kind="bar")
plt.title("Members by Type")
plt.show()